In [ ]:
# Lecture 10 — DAGs, Jobs, Stages & Tasks
# Spark 101 · Module 3 · Lecture 10
#
# Import this file into a Databricks workspace as a notebook.
# The demo runs the query — the payoff is switching to the Spark UI and walking
# the hierarchy: Jobs → Stages → Tasks.

In [ ]:
# Partition counts, serverless-safe.
# df.rdd.getNumPartitions() uses the RDD API, which is BLOCKED on serverless
# compute. We count non-empty partitions via spark_partition_id() instead.
from pyspark.sql.functions import spark_partition_id

def part_count(df):
    return df.select(spark_partition_id()).distinct().count()

In [ ]:
# Load the sample dataset — multi-GB, real Parquet, many partitions.
df = spark.read.table("samples.nyctaxi.trips")
print(f"Partitions: {part_count(df)}")

In [ ]:
# Build a multi-stage query. Two wide transformations → two shuffles → three Stages.
from pyspark.sql.functions import col

result = (
    df.filter(col("trip_distance") > 5)      # narrow
      .groupBy("pickup_zip").count()    # WIDE — shuffle 1
      .orderBy(col("count").desc())          # WIDE — shuffle 2
)

# Zero jobs so far — nothing has run yet.
print("Plan built. 0 jobs. Now flip to Spark UI and confirm.")

In [ ]:
# The action. Fires everything. Flip to the Spark UI as soon as this cell finishes.
# Look for: 1 Job · 3 Stages · task counts per stage.
result.show()

In [ ]:
# A second action on the same DataFrame — a second Job.
# This is the L8 trap live: same plan runs again.
print(f"Row count: {result.count()}")

In [ ]:
# In the Spark UI now:
#   Jobs tab                → 2 Jobs (one per action)
#   Click a Job             → DAG viz with 3 Stage boxes + shuffle exchanges
#   Stages tab              → Tasks column shows partition count per stage
#   Click a Stage           → per-task duration, input size, shuffle read/write
#   Look for                → skew (one task much slower than the rest)

In [ ]:
# Narrow-only query: filter + select = ONE stage, no shuffle.
from pyspark.sql.functions import col
narrow = df.filter(col("trip_distance") > 2).select("pickup_zip", "fare_amount")
narrow.explain()     # no Exchange -> single stage
narrow.count()

In [ ]:
# Add distinct() — another wide op, another shuffle, another stage.
multi = (df.groupBy("pickup_zip").count()
           .orderBy(col("count").desc())
           .distinct())
multi.explain()      # count the Exchange nodes = number of shuffles
multi.show()

In [ ]:
# After running these, the Spark UI Jobs tab shows one Job per action above.
print("Flip to the Spark UI — compare stage counts across the queries.")